<img src="./uva_seal.png">  

## PySpark Hotspot Demo

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: March 10, 2026

---  


### BACKGROUND

In Spark, a data hotspot happens when one or a few keys have much more data than others — causing one task to do most of the work while others sit idle.

It is a data imbalance problem.

This notebook demonstrates two small examples of how it arises and how it can be remediated.

#### EXAMPLE 1: SALTING A SMALLER DATASET

In [40]:
# Import modules
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

# Create Spark session
spark = SparkSession.builder.appName("HotspotDemo").master("local[*]").getOrCreate()

In [41]:
# check number of cores
spark.sparkContext.defaultParallelism

4

In [42]:
# Create skewed data
data = [
    ("user1", "click"), ("user1", "click"), ("user1", "click"), ("user1", "click"), ("user1", "click"),
    ("user1", "click"), ("user1", "click"), ("user1", "click"), ("user1", "click"), ("user1", "click"), # Hot key
    ("user2", "click"), ("user3", "click"), ("user4", "click")
]

df = spark.createDataFrame(data, ["user_id", "event"])

# Aggregate by user_id which causes data skew on "user1"
counts = df.groupBy("user_id").agg(count("*").alias("total_events"))

counts.show()

[Stage 0:>                                                          (0 + 4) / 4]

+-------+------------+
|user_id|total_events|
+-------+------------+
|  user1|          10|
|  user3|           1|
|  user2|           1|
|  user4|           1|
+-------+------------+



#### THE PROBLEM

Most records belong to "user1" key.

During `groupBy`, that key becomes a hot partition.

Spark assigns a single task to handle all "user1" data, slowing the job.

#### A SOLUTION: SALTING

Have Spark add a small random “salt” (0–9) to each record’s key, spreading the hot key’s data across multiple reducers.

Then aggregate back to the original key after processing.

In [116]:
from pyspark.sql.functions import lit, concat, rand, expr

# Add a random salt to break up the skewed key
salted = df.withColumn("salted_key", concat(col("user_id"), lit("_"), (rand() * 10).cast("int")))
print('Salted data:')
salted.show()

# Aggregate by salted key
salted_counts = salted.groupBy("salted_key").agg(count("*").alias("partial_count"))
print('Salted counts by key:')
salted_counts.show()

# Aggregate back by original user_id
final_counts = salted_counts.groupBy(expr("split(salted_key, '_')[0]").alias("user_id")) \
                            .agg(expr("sum(partial_count)").alias("total_events"))

print('Counts by original key:')
final_counts.show()


Salted data:
+--------+--------+----------+
| user_id|event_id|salted_key|
+--------+--------+----------+
|hot_user|       0|hot_user_8|
|hot_user|       1|hot_user_0|
|hot_user|       2|hot_user_6|
|hot_user|       3|hot_user_4|
|hot_user|       4|hot_user_3|
|hot_user|       5|hot_user_6|
|hot_user|       6|hot_user_1|
|hot_user|       7|hot_user_0|
|hot_user|       8|hot_user_6|
|hot_user|       9|hot_user_8|
|hot_user|      10|hot_user_5|
|hot_user|      11|hot_user_1|
|hot_user|      12|hot_user_8|
|hot_user|      13|hot_user_4|
|hot_user|      14|hot_user_3|
|hot_user|      15|hot_user_0|
|hot_user|      16|hot_user_5|
|hot_user|      17|hot_user_2|
|hot_user|      18|hot_user_8|
|hot_user|      19|hot_user_9|
+--------+--------+----------+
only showing top 20 rows
Salted counts by key:
+----------+-------------+
|salted_key|partial_count|
+----------+-------------+
|hot_user_5|            6|
|hot_user_2|            6|
|hot_user_4|            9|
|hot_user_3|           10|
|hot_us

---

#### EXAMPLE 2: LARGER EXAMPLE OF SALTING WITH RUNTIME COMPARE

First, we create a larger, skewed dataset

In [127]:
from pyspark.sql.functions import split
import time

# Create Spark session
spark = SparkSession.builder.appName("HotspotTimingDemo").getOrCreate()

# Most rows belong to one key ("hot_user")
data = [("hot_user", i) for i in range(100)] + \
       [(f"user_{i}", i) for i in range(1, 3)]

df = spark.createDataFrame(data, ["user_id", "event_id"])

print(f"Total rows: {df.count():,}")

Total rows: 102


In [128]:
# inspect some rows
df.head(5)

[Row(user_id='hot_user', event_id=0),
 Row(user_id='hot_user', event_id=1),
 Row(user_id='hot_user', event_id=2),
 Row(user_id='hot_user', event_id=3),
 Row(user_id='hot_user', event_id=4)]

**Next, we aggregate without fixing skew and compute runtime**

In [129]:
start_time = time.time()
counts_no_fix = df.groupBy("user_id").agg(count("*").alias("total_events"))
counts_no_fix.show()  # force computation
time_no_fix = time.time() - start_time
print(f"Runtime without fix: {time_no_fix:.2f} seconds")

+--------+------------+
| user_id|total_events|
+--------+------------+
|hot_user|         100|
|  user_1|           1|
|  user_2|           1|
+--------+------------+

Runtime without fix: 0.26 seconds


**Salt the data to break up skew**

In [130]:
salted = df.withColumn("salted_key", concat(col("user_id"), lit("_"), (rand() * 10).cast("int")))

salted_counts = salted.groupBy("salted_key").agg(count("*").alias("partial_count"))

final_counts = salted_counts.groupBy(expr("split(salted_key, '_')[0]").alias("user_id")) \
                            .agg(expr("sum(partial_count)").alias("total_events"))

**Compute and compare runtimes**

In [131]:
start_time = time.time()
final_counts.show()  # force computation
time_with_fix = time.time() - start_time
print(f"Runtime with salting fix: {time_with_fix:.2f} seconds")

# compare runtimes
improvement = ((time_no_fix - time_with_fix) / time_no_fix) * 100
print(f"Improvement: {improvement:.1f}% faster after removing hotspot")

#spark.stop()

+-------+------------+
|user_id|total_events|
+-------+------------+
|    hot|         100|
|   user|           2|
+-------+------------+

Runtime with salting fix: 0.23 seconds
Improvement: 10.6% faster after removing hotspot


**NOTE**: Given the shuffling of the data, salting may not speed up runtime in all cases, but it improves data skew